# YG_utils_analysis - Examples Notebook

This notebook demonstrates common usage patterns for the YG_utils_analysis package with realistic data processing pipelines.

## Table of Contents

1. [Setup and Installation](#setup)
2. [Fiber Photometry Pipeline](#fp-pipeline)
3. [Behavioral Video Processing](#video-processing)
4. [DeepLabCut Analysis](#dlc-analysis)
5. [Epoch Analysis](#epoch-analysis)
6. [Group-Level Analysis](#group-analysis)
7. [Visualization Examples](#visualization)

<a id='setup'></a>
## 1. Setup and Installation

First, make sure the package is installed:

In [ ]:
# If not installed, install the package
# !pip install git+https://github.com/parkgilbong/YG_utils_analysis.git

# Import necessary modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("Setup complete!")

<a id='fp-pipeline'></a>
## 2. Fiber Photometry Processing Pipeline

### Example 2.1: Single-Channel FP Preprocessing

In [ ]:
from utils.FPFunctions import FP_preprocessing_1ch
from utils.FileFunctions import ensure_dir

# Define paths
tank_path = '/path/to/TDT/tank'  # Replace with your actual path
output_folder = ensure_dir('/path/to/output/preprocessed')

# Note: This is a demonstration. Uncomment to run with real data.
# FP_preprocessing_1ch(
#     Tank_path=tank_path,
#     Dest_folder=output_folder,
#     sys='tdt',
#     Detrending_method='Exp_fit',
#     Use_CamTick=True,
#     FPS=25,
#     Rec_duration=600,
#     SaveAsCSV=True
# )

print("FP preprocessing complete! Check output folder for results.")

### Example 2.2: Visualize Preprocessed FP Data

In [ ]:
from utils.PlotFunctions import plot_single_line

# Generate synthetic FP data for demonstration
np.random.seed(42)
time = np.linspace(0, 600, 15000)  # 10 minutes at ~25 Hz
baseline = 0.1
trend = 0.02 * np.exp(-time / 200)  # Exponential decay
signal = baseline + trend + 0.3 * np.sin(time / 30) + 0.1 * np.random.randn(len(time))

# Plot raw signal
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Raw signal
plot_single_line(
    x=time,
    y=signal,
    fig_size=None,
    fig_title='Raw FP Signal',
    x_label='Time (s)',
    y_label='Fluorescence (a.u.)',
    x_lim=(0, 600),
    y_lim=None,
    color='gray',
    ax=axes[0]
)

# Detrended dF/F signal (simulated)
dff = (signal - baseline - trend) / (baseline + trend)
z_scored = (dff - np.mean(dff)) / np.std(dff)

plot_single_line(
    x=time,
    y=z_scored,
    fig_size=None,
    fig_title='Processed dF/F (z-scored)',
    x_label='Time (s)',
    y_label='dF/F (z-score)',
    x_lim=(0, 600),
    y_lim=(-3, 3),
    color='green',
    ax=axes[1]
)

plt.tight_layout()
plt.show()

print("Signal processing visualization complete.")

<a id='video-processing'></a>
## 3. Behavioral Video Processing

### Example 3.1: Extract Frames from Video

In [ ]:
from utils.VideoFunctions import extract_frames
from utils.FileFunctions import ensure_dir

# Define video and output paths
video_path = '/path/to/behavioral_video.mp4'
output_folder = ensure_dir('/path/to/output/frames')

# Extract specific frames for figure panels
frame_indices = [0, 750, 1500, 2250, 3000]  # Every 30 seconds at 25 FPS

# Note: Uncomment to run with real video
# extract_frames(
#     video_path=video_path,
#     frame_indices=frame_indices,
#     output_folder=output_folder
# )

print(f"Would extract {len(frame_indices)} frames to {output_folder}")

### Example 3.2: Resize Videos for Sharing

In [ ]:
from utils.VideoFunctions import resize_video

# Reduce video size to 50% for presentations
input_video = '/path/to/large_video.mp4'
output_video = '/path/to/small_video.mp4'

# Note: Uncomment to run with real video
# resize_video(
#     input_path=input_video,
#     output_path=output_video,
#     scale_factor=0.5
# )

print("Video resizing reduces file size by ~75% while maintaining quality.")

<a id='dlc-analysis'></a>
## 4. DeepLabCut Data Analysis

### Example 4.1: Load and Process DLC Output

In [ ]:
from utils.DLCFunctions import df_to_dic_single, get_velocity

# Simulate DLC output data
n_frames = 15000
bodyparts = ['Nose', 'Center', 'TailBase']

# Create synthetic tracking data
dlc_data = {}
for bp in bodyparts:
    # Simulate movement in a circular pattern
    t = np.linspace(0, 4*np.pi, n_frames)
    dlc_data[bp] = {
        'x': 320 + 100 * np.cos(t) + 5 * np.random.randn(n_frames),
        'y': 240 + 100 * np.sin(t) + 5 * np.random.randn(n_frames),
        'likelihood': 0.95 + 0.05 * np.random.randn(n_frames)
    }
    dlc_data[bp]['likelihood'] = np.clip(dlc_data[bp]['likelihood'], 0, 1)

print(f"Simulated DLC data for {len(bodyparts)} body parts, {n_frames} frames")

# Calculate velocity
velocity_result = get_velocity(
    DLCresult=dlc_data,
    bpt='Center',
    FPS=25,
    pcutoff=0.9
)

print(f"Mean velocity: {np.mean(velocity_result['velocity']):.2f} pixels/s")

### Example 4.2: Visualize Tracking and Velocity

In [ ]:
from utils.PlotFunctions import plot_dual_line

# Create trajectory plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot trajectory
axes[0].plot(dlc_data['Center']['x'], dlc_data['Center']['y'], 
             'b-', alpha=0.3, linewidth=0.5)
axes[0].plot(dlc_data['Center']['x'][0], dlc_data['Center']['y'][0], 
             'go', markersize=10, label='Start')
axes[0].plot(dlc_data['Center']['x'][-1], dlc_data['Center']['y'][-1], 
             'ro', markersize=10, label='End')
axes[0].set_xlabel('X (pixels)')
axes[0].set_ylabel('Y (pixels)')
axes[0].set_title('Animal Trajectory')
axes[0].legend()
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Plot velocity over time
time = np.arange(len(velocity_result['velocity'])) / 25
axes[1].plot(time, velocity_result['velocity'], 'k-', linewidth=0.5)
axes[1].axhline(np.mean(velocity_result['velocity']), 
                color='r', linestyle='--', label='Mean velocity')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Velocity (pixels/s)')
axes[1].set_title('Movement Velocity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

<a id='epoch-analysis'></a>
## 5. Event-Centered Epoch Analysis

### Example 5.1: Extract Peri-Event Traces

In [ ]:
from utils.FPFunctions import extract_traces_with_padding

# Simulate behavioral events
event_times = [(50, 55), (150, 160), (250, 258), (350, 365), (450, 470)]

# Generate FP signal with events
time = np.linspace(0, 600, 15000)
signal = np.random.randn(len(time)) * 0.3  # Baseline noise

# Add calcium transients at event times
for onset, offset in event_times:
    event_indices = (time >= onset) & (time <= offset + 3)
    t_event = time[event_indices] - onset
    # Add calcium response (rise and decay)
    signal[event_indices] += 2.5 * (1 - np.exp(-t_event / 1.0)) * np.exp(-t_event / 3.0)

# Extract peri-event traces
epoch_traces = extract_traces_with_padding(
    signal=signal,
    time=time,
    time_tuples=event_times,
    pre_window_sec=5.0,
    post_window_sec=10.0,
    FPS=25,
    align_to='onset'
)

print(f"Extracted {len(epoch_traces)} peri-event traces")
print(f"Each trace has {epoch_traces[0].shape} timepoints")

### Example 5.2: Visualize Epoch Analysis

In [ ]:
from utils.PlotFunctions import plot_traces_with_mean, plot_trace_heatmap

# Stack traces into array
all_traces = np.vstack([trace.reshape(1, -1) for trace in epoch_traces])
epoch_time = np.linspace(-5, 10, all_traces.shape[1])

# Create figure with multiple panels
fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# Panel A: Full time series with events marked
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(time, signal, 'g-', linewidth=0.5, alpha=0.7)
for onset, offset in event_times:
    ax1.axvspan(onset, offset, alpha=0.2, color='red')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('dF/F (z-score)')
ax1.set_title('A. Full Recording with Behavioral Events')
ax1.set_xlim(0, 600)
ax1.grid(True, alpha=0.3)

# Panel B: Individual trials
ax2 = fig.add_subplot(gs[1, 0])
for i, trace in enumerate(all_traces):
    ax2.plot(epoch_time, trace, alpha=0.5, linewidth=1)
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='Event Onset')
ax2.set_xlabel('Time from Event (s)')
ax2.set_ylabel('dF/F (z-score)')
ax2.set_title('B. Individual Trials')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Panel C: Mean ± SEM
ax3 = fig.add_subplot(gs[1, 1])
plot_traces_with_mean(
    trace_array=all_traces,
    trace_time=epoch_time,
    ax=ax3,
    color='blue',
    title='C. Average Response',
    xlabel='Time from Event (s)',
    ylabel='dF/F (z-score)',
    mode='sem'
)
ax3.axvline(0, color='red', linestyle='--', linewidth=2)
ax3.axhline(0, color='gray', linestyle='-', linewidth=0.5)

# Panel D: Heatmap
ax4 = fig.add_subplot(gs[2, :])
plot_trace_heatmap(
    traces=all_traces,
    trace_time=epoch_time,
    vmin=-1,
    vmax=3,
    ax=ax4,
    title='D. Trial-by-Trial Heatmap',
    xlabel='Time from Event (s)',
    ylabel='Trial #',
    cmap='RdYlBu_r'
)
ax4.axvline(0, color='white', linestyle='--', linewidth=2)

plt.savefig('epoch_analysis_example.png', dpi=300, bbox_inches='tight')
plt.show()

print("Epoch analysis visualization complete!")

<a id='group-analysis'></a>
## 6. Group-Level Analysis

### Example 6.1: Compare Multiple Animals

In [ ]:
from utils.PlotFunctions import plot_multi_line

# Simulate data from multiple animals
n_animals_per_group = 5
time_vec = np.linspace(-5, 10, 375)

# Control group
control_traces = []
for i in range(n_animals_per_group):
    t_shifted = time_vec + np.random.randn() * 0.2
    trace = 0.5 * np.exp(-((t_shifted - 0.5) ** 2) / 2) + 0.1 * np.random.randn(len(time_vec))
    control_traces.append(trace)
control_traces = np.array(control_traces)

# Treatment group (enhanced response)
treatment_traces = []
for i in range(n_animals_per_group):
    t_shifted = time_vec + np.random.randn() * 0.2
    trace = 1.5 * np.exp(-((t_shifted - 0.5) ** 2) / 2) + 0.1 * np.random.randn(len(time_vec))
    treatment_traces.append(trace)
treatment_traces = np.array(treatment_traces)

# Calculate means and SEMs
control_mean = np.mean(control_traces, axis=0)
control_sem = np.std(control_traces, axis=0) / np.sqrt(n_animals_per_group)
treatment_mean = np.mean(treatment_traces, axis=0)
treatment_sem = np.std(treatment_traces, axis=0) / np.sqrt(n_animals_per_group)

# Plot comparison
xy_pairs = [
    (time_vec, control_mean, 'Control'),
    (time_vec, treatment_mean, 'Treatment')
]
sem_pairs = [
    (time_vec, control_sem),
    (time_vec, treatment_sem)
]

plot_multi_line(
    xy_pairs=xy_pairs,
    sem_pairs=sem_pairs,
    fig_size=(10, 6),
    title='Group Comparison: Control vs Treatment',
    x_label='Time from Event (s)',
    y_label='dF/F (z-score)',
    colors=['gray', 'red'],
    save=True
)

plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
plt.grid(True, alpha=0.3)
plt.savefig('group_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Group comparison complete!")

### Example 6.2: Calculate Summary Statistics

In [ ]:
from utils.FPFunctions import calculate_auc
from scipy import stats

# Calculate AUC for each animal during response period (0-5s)
response_window = (time_vec >= 0) & (time_vec <= 5)

control_aucs = []
for trace in control_traces:
    auc = np.trapz(trace[response_window], time_vec[response_window])
    control_aucs.append(auc)

treatment_aucs = []
for trace in treatment_traces:
    auc = np.trapz(trace[response_window], time_vec[response_window])
    treatment_aucs.append(auc)

# Statistical comparison
t_stat, p_value = stats.ttest_ind(control_aucs, treatment_aucs)

# Create summary table
summary_df = pd.DataFrame({
    'Group': ['Control', 'Treatment'],
    'N': [len(control_aucs), len(treatment_aucs)],
    'Mean AUC': [np.mean(control_aucs), np.mean(treatment_aucs)],
    'SEM': [np.std(control_aucs) / np.sqrt(len(control_aucs)), 
            np.std(treatment_aucs) / np.sqrt(len(treatment_aucs))]
})

print("\nSummary Statistics:")
print(summary_df.to_string(index=False))
print(f"\nStatistical Test:")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(6, 6))
positions = [1, 2]
bp = ax.boxplot([control_aucs, treatment_aucs], positions=positions,
                 labels=['Control', 'Treatment'], patch_artist=True)
bp['boxes'][0].set_facecolor('lightgray')
bp['boxes'][1].set_facecolor('lightcoral')

# Add individual points
for i, aucs in enumerate([control_aucs, treatment_aucs]):
    x = np.random.normal(positions[i], 0.04, size=len(aucs))
    ax.scatter(x, aucs, alpha=0.6, s=50, c='black')

ax.set_ylabel('AUC (0-5s)')
ax.set_title('Response Magnitude Comparison')
ax.grid(True, alpha=0.3)

# Add significance indicator
if p_value < 0.05:
    y_max = max(max(control_aucs), max(treatment_aucs))
    ax.plot([1, 2], [y_max * 1.1, y_max * 1.1], 'k-')
    sig_label = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*'
    ax.text(1.5, y_max * 1.15, sig_label, ha='center', fontsize=14)

plt.tight_layout()
plt.savefig('auc_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

<a id='visualization'></a>
## 7. Advanced Visualization Examples

### Example 7.1: Publication-Quality Figure

In [ ]:
# Create a complete publication figure
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.4)

# Panel A: Example trace with events
ax_a = fig.add_subplot(gs[0, :])
ax_a.plot(time[:3000], signal[:3000], 'g-', linewidth=1)
for onset, offset in event_times[:2]:
    if onset < 120:
        ax_a.axvspan(onset, offset, alpha=0.2, color='red', label='Event' if onset == event_times[0][0] else '')
ax_a.set_xlabel('Time (s)', fontsize=12)
ax_a.set_ylabel('dF/F (z-score)', fontsize=12)
ax_a.set_title('A. Representative Recording', fontsize=14, fontweight='bold')
ax_a.legend(fontsize=10)
ax_a.grid(True, alpha=0.3)

# Panel B: Peri-event traces
ax_b = fig.add_subplot(gs[1, 0])
for i, trace in enumerate(all_traces):
    ax_b.plot(epoch_time, trace, alpha=0.4, linewidth=1, color='blue')
ax_b.axvline(0, color='red', linestyle='--', linewidth=2)
ax_b.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax_b.set_xlabel('Time from Event (s)', fontsize=12)
ax_b.set_ylabel('dF/F (z-score)', fontsize=12)
ax_b.set_title('B. Individual Trials', fontsize=14, fontweight='bold')
ax_b.grid(True, alpha=0.3)

# Panel C: Average response
ax_c = fig.add_subplot(gs[1, 1])
mean_trace = np.mean(all_traces, axis=0)
sem_trace = np.std(all_traces, axis=0) / np.sqrt(len(all_traces))
ax_c.plot(epoch_time, mean_trace, 'b-', linewidth=2, label='Mean')
ax_c.fill_between(epoch_time, mean_trace - sem_trace, mean_trace + sem_trace, 
                   alpha=0.3, color='blue', label='SEM')
ax_c.axvline(0, color='red', linestyle='--', linewidth=2)
ax_c.axhline(0, color='gray', linestyle='-', linewidth=0.5)
ax_c.set_xlabel('Time from Event (s)', fontsize=12)
ax_c.set_ylabel('dF/F (z-score)', fontsize=12)
ax_c.set_title('C. Mean ± SEM', fontsize=14, fontweight='bold')
ax_c.legend(fontsize=10)
ax_c.grid(True, alpha=0.3)

# Panel D: Heatmap
ax_d = fig.add_subplot(gs[1, 2])
im = ax_d.imshow(all_traces, aspect='auto', cmap='RdYlBu_r', 
                 extent=[epoch_time[0], epoch_time[-1], len(all_traces), 0],
                 vmin=-1, vmax=3)
ax_d.axvline(0, color='white', linestyle='--', linewidth=2)
ax_d.set_xlabel('Time from Event (s)', fontsize=12)
ax_d.set_ylabel('Trial #', fontsize=12)
ax_d.set_title('D. Trial Heatmap', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax_d, label='dF/F (z-score)')

# Panel E: Group comparison
ax_e = fig.add_subplot(gs[2, :2])
ax_e.plot(time_vec, control_mean, 'gray', linewidth=2, label='Control')
ax_e.fill_between(time_vec, control_mean - control_sem, control_mean + control_sem,
                  alpha=0.3, color='gray')
ax_e.plot(time_vec, treatment_mean, 'red', linewidth=2, label='Treatment')
ax_e.fill_between(time_vec, treatment_mean - treatment_sem, treatment_mean + treatment_sem,
                  alpha=0.3, color='red')
ax_e.axvline(0, color='black', linestyle='--', linewidth=1)
ax_e.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)
ax_e.set_xlabel('Time from Event (s)', fontsize=12)
ax_e.set_ylabel('dF/F (z-score)', fontsize=12)
ax_e.set_title('E. Group Comparison', fontsize=14, fontweight='bold')
ax_e.legend(fontsize=10)
ax_e.grid(True, alpha=0.3)

# Panel F: Quantification
ax_f = fig.add_subplot(gs[2, 2])
bp = ax_f.boxplot([control_aucs, treatment_aucs], positions=[1, 2],
                   labels=['Control', 'Treatment'], patch_artist=True,
                   widths=0.6)
bp['boxes'][0].set_facecolor('lightgray')
bp['boxes'][1].set_facecolor('lightcoral')
for i, aucs in enumerate([control_aucs, treatment_aucs]):
    x = np.random.normal([1, 2][i], 0.04, size=len(aucs))
    ax_f.scatter(x, aucs, alpha=0.6, s=50, c='black')
ax_f.set_ylabel('AUC (a.u.)', fontsize=12)
ax_f.set_title('F. Response Magnitude', fontsize=14, fontweight='bold')
ax_f.grid(True, alpha=0.3, axis='y')

plt.savefig('complete_figure.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Publication-quality figure created successfully!")

## Summary

This notebook demonstrated:

1. ✅ FP data preprocessing and visualization
2. ✅ Video processing workflows
3. ✅ DLC tracking analysis
4. ✅ Event-centered epoch analysis
5. ✅ Group-level statistical comparisons
6. ✅ Publication-quality visualizations

### Next Steps

- Replace synthetic data with your actual experimental data
- Adjust parameters based on your specific experimental conditions
- See the [cheatsheets](README.md#documentation--cheatsheets) for more examples
- Check [utils_summary.md](utils_summary.md) for complete function reference

### Additional Resources

- **[FPFunctions Cheatsheet](cheatsheet_FPFunctions.md)** - Fiber photometry processing
- **[PlotFunctions Cheatsheet](cheatsheet_PlotFunctions.md)** - Visualization tools
- **[VideoFunctions Cheatsheet](cheatsheet_VideoFunctions.md)** - Video processing
- **[DLCFunctions Cheatsheet](cheatsheet_DLCFunctions.md)** - Tracking analysis
- **[FileFunctions Cheatsheet](cheatsheet_FileFunctions.md)** - File operations